# The EM2C combustor: a cross-code check against OSCILOS

Nefes solves the mean flow and the linear thermoacoustics of the *same* network from the *same* residuals.
A published case that fixes only the geometry, the operating point, the boundary reflections and the flame transfer function therefore exercises the whole chain at once, and it can be checked against an independent code that solves the acoustics alone on a mean flow of its own.

This notebook reproduces the **stable configuration of the EM2C swirl-stabilized combustor** of Palies et al. [1], as it is set up in Sec. 5.4.1 of the OSCILOS technical report [2].
That case is stated entirely in text — lengths, radii, temperatures, the mean velocity, the reflection coefficients, and a closed-form flame transfer function — so nothing has to be read off a figure, and OSCILOS reports the dominant mode of the combustor as

$$
f = 152.6\;\mathrm{Hz}, \qquad \sigma = -19.1\;\mathrm{s^{-1}},
$$

a decaying, and therefore stable, mode.

## The case

The combustor is reduced to three cylinders: a plenum, an injection unit, and a combustion chamber open to the atmosphere, with a compact flame anchored at the chamber inlet.

| | plenum | injection unit | chamber |
|---|---|---|---|
| length [mm] | 117.3 | 117 | 100 |
| radius [mm] | 32.5 | 10.585 | 35 |

The reactants enter at $\overline{p} = 1\,$bar and $\overline{T} = 300\,$K, the mean velocity at the injector exit is $4.13\,\mathrm{m/s}$, and the burnt gas sits at $1600\,$K.
The inlet is rigid ($R = +1$) and the outlet is pressure-release ($R = -1$).
The flame responds to the velocity fluctuation in the injection unit through a second-order low-pass transfer function with a pure time lag [3],

$$
\mathcal{F}(\omega) \;=\; \frac{\omega_c^{2}}{s^{2} + 2\zeta\omega_c s + \omega_c^{2}}\,e^{-s\tau},
\qquad s = \mathrm{i}\omega,
$$

with cutoff $f_c = \omega_c/2\pi = 200\,$Hz, damping ratio $\zeta = 0.5$ and lag $\tau = 2\,$ms.

## Matching the reference's conventions

Two choices in the reference must be matched for the comparison to be like-for-like.

* Its mean flow runs on a **calorically perfect gas** with $\gamma = 1.4$ and $R = \mathcal{R}_u/W_{\mathrm{air}}$ on *both* sides of the flame (its "constant $\gamma$" option), which is exactly the Nefes `perfect_gas` model.
* Its flame sits at an **abrupt area increase**, split into a Borda–Carnot expansion followed by constant-area heat addition, which is a `sudden_area_change` followed by a `heat_release_flame`.

Everything else — the mean flow, the linearization, the boundary closures — Nefes derives itself.

We will (1) solve the mean flow, (2) attach the flame transfer function, (3) pin the mode and compare it against the published pair, (4) find out where the damping comes from, (5) show that the entropy wave is a spectator at this outlet, and (6) sweep the flame lag to see where this configuration sits relative to the instability band.

In [ ]:
import os, sys, warnings

_root = os.getcwd()
while not os.path.isdir(os.path.join(_root, "nefes")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)

import numpy as np
import plotly.graph_objects as go

from nefes.assembly.recover import ES_C, ES_M, ES_P, ES_T, ES_U
from nefes.elements import catalog as cat
from nefes.elements.dynamic_source import heat_release_response, n_tau_lowpass2
from nefes.perturbation import eigenmodes
from nefes.perturbation.operator.boundary_bc import PerturbationBC
from nefes.plotting import use_nefes_theme, plot_transfer_function
from nefes.shell import Network
from nefes.thermo.configure import perfect_gas

# Below is because plotly LaTeX rendering is not working out of box in notebooks viewed in VSCode/Cursor.
from IPython.display import display, HTML
import plotly.offline as pyo
pyo.init_notebook_mode()
display(HTML(
    '<script src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.7/MathJax.js?config=TeX-AMS-MML_SVG"></script>'
))

_ = use_nefes_theme()

## 1. The mean flow

The reference's air is a perfect gas with $\gamma = 1.4$ and $R = \mathcal{R}_u / W_{\mathrm{air}} = 287.05\;\mathrm{J/(kg\,K)}$, held on both sides of the flame.

The mean flow is not prescribed section by section: we fix the mass flow that puts $4.13\;\mathrm{m/s}$ at the injector exit, fix the heat power that raises the burnt gas to $1600\,$K, and let the solver find the rest.
The plenum contraction is smooth and lossless, so it is an `isentropic_area_change`; the chamber inlet is an abrupt dump, so it is a `sudden_area_change`, which carries the Borda–Carnot total-pressure loss.
The compact flame follows immediately after, at the chamber area.

The flame transfer function is attached to the flame element and read on the edge that leaves the injection unit, which is where the reference's $\hat u/\overline{u}$ is measured.
The steady solve ignores it: a constant mean heat release is acoustically passive, so the mean flow is the same with and without the flame's dynamic response.

In [ ]:
# OSCILOS's air: R = R_u / W_air, and gamma held at 1.4 on both sides of the flame
R_AIR = 8.3145 / 28.96512 * 1000.0
GAMMA = 1.4
CP = GAMMA * R_AIR / (GAMMA - 1.0)

# Plenum / injection unit / combustion chamber
LENGTHS = (0.1173, 0.117, 0.100)
RADII = (0.0325, 0.010585, 0.035)
AREAS = tuple(np.pi * r**2 for r in RADII)

# Operating point
P_MEAN = 1.0e5      # mean pressure [Pa]
T_COLD = 300.0      # reactant temperature [K]
T_BURNT = 1600.0    # burnt-gas temperature [K]
U_INJECTOR = 4.13   # mean velocity at the injector exit [m/s]

# Flame transfer function: unit gain at zero frequency, cutoff, damping ratio, time lag
FTF_GAIN, FTF_FC, FTF_ZETA, FTF_TAU = 1.0, 200.0, 0.5, 2.0e-3

# The eigenvalue OSCILOS reports for the dominant mode
OSCILOS_FREQ_HZ, OSCILOS_GROWTH = 152.6, -19.1


def em2c(tau=FTF_TAU, *, T_cold=T_COLD, active=True, expansion="borda"):
    """Build and solve the three-cylinder EM2C combustor.

    The topology is spelled out node by node: rigid inlet -> plenum -> contraction ->
    injection unit -> dump -> compact flame -> chamber -> open end.  The flame's velocity
    reference is the edge leaving the injection unit; rather than guess its index up front,
    we take it from the value ``connect`` returns and attach the transfer function afterwards
    with ``set_dynamic_source``.

    Parameters
    ----------
    tau : float, optional
        Flame time lag [s] of the transfer function.
    T_cold : float, optional
        Reactant temperature [K] at the plenum inlet.
    active : bool, optional
        Attach the flame transfer function (default ``True``).  With ``False`` the flame is a
        steady heat source, and the network is acoustically passive.
    expansion : {"borda", "isentropic"}, optional
        Model of the abrupt area increase at the chamber inlet.  ``"borda"`` (default) is the
        dissipative sudden expansion; ``"isentropic"`` replaces it with a lossless area change,
        removing the interior sink of acoustic energy.

    Returns
    -------
    Solution
        The converged mean-flow solution (carrying ``.problem``, ``.x``, ``.table()``).
    """
    rho_injector = P_MEAN / (R_AIR * T_cold)
    mdot = rho_injector * U_INJECTOR * AREAS[1]
    # heat power for the prescribed burnt temperature; the kinetic terms are O(1e-5) of it here
    qdot = mdot * CP * (T_BURNT - T_cold)

    net = Network(perfect_gas(R_AIR, GAMMA), p_ref=P_MEAN, T_ref=T_cold, mdot_ref=mdot)

    i_inlet = net.add(cat.mass_flow_inlet(mdot, T_cold, perturbation_bc=PerturbationBC.hard_wall()))
    i_plenum = net.add(cat.duct(LENGTHS[0], name="plenum"))
    i_contract = net.add(cat.isentropic_area_change(name="contraction"))
    i_injector = net.add(cat.duct(LENGTHS[1], name="injector"))
    i_dump = net.add(
        cat.sudden_area_change(name="dump") if expansion == "borda" else cat.isentropic_area_change(name="dump")
    )
    i_flame = net.add(cat.heat_release_flame(qdot))
    i_chamber = net.add(cat.duct(LENGTHS[2], name="chamber"))
    i_outlet = net.add(cat.pressure_outlet(P_MEAN, perturbation_bc=PerturbationBC.open_end()))

    net.connect(i_inlet, i_plenum, AREAS[0])
    net.connect(i_plenum, i_contract, AREAS[0])
    net.connect(i_contract, i_injector, AREAS[1])
    e_ref = net.connect(i_injector, i_dump, AREAS[1])   # the flame's velocity reference
    net.connect(i_dump, i_flame, AREAS[2])
    net.connect(i_flame, i_chamber, AREAS[2])
    net.connect(i_chamber, i_outlet, AREAS[2])

    if active:
        ftf = n_tau_lowpass2(FTF_GAIN, tau, FTF_FC, FTF_ZETA)
        net.set_dynamic_source(i_flame, heat_release_response(ftf, ref_edge=e_ref))

    sol = net.solve()
    assert sol.converged
    return sol


sol = em2c()
sol.print_states()

Edge 3 leaves the injection unit at the prescribed $4.13\,\mathrm{m/s}$; edge 4 is the same flow after the dump, slowed by the area ratio; edges 5–6 carry the burnt gas at $1600\,$K.
The mean pressure never departs from $1\,$bar by more than about $10\,$Pa, so the flow is firmly low-Mach ($M \lesssim 0.012$) and the mean state is essentially the reference's.

In [ ]:
# The mean state along the axis: piecewise constant within each cylinder, jumping at the
# contraction and at the dump/flame plane (edge 1 = plenum, 3 = injector, 6 = chamber).
est = sol.table()
x_face = np.array([0.0, LENGTHS[0], LENGTHS[0] + LENGTHS[1], sum(LENGTHS)])
segments = [(x_face[i], x_face[i + 1], e) for i, e in enumerate((1, 3, 6))]

fig = go.Figure()
for var, row, name, axis in (("u", ES_U, "mean velocity", "y"), ("T", ES_T, "mean temperature", "y2")):
    xs, ys = [], []
    for x0, x1, e in segments:
        xs += [x0, x1, None]
        ys += [est[row, e], est[row, e], None]
    fig.add_scatter(x=xs, y=ys, mode="lines", name=name, yaxis=axis,
                    line=dict(width=2.5, dash="solid" if var == "u" else "dot"))
fig.add_vline(x=x_face[1], line_width=1, line_dash="dash", line_color="#9aa5b1",
              annotation_text="contraction", annotation_position="top")
fig.add_vline(x=x_face[2], line_width=1, line_dash="dash", line_color="#d62728",
              annotation_text="dump + flame", annotation_position="top")
fig.update_layout(
    title="Solved mean flow along the combustor",
    xaxis_title=r"$x\;[\mathrm{m}]$",
    yaxis=dict(title=r"$\overline{u}\;[\mathrm{m/s}]$"),
    yaxis2=dict(title=r"$\overline{T}\;[\mathrm{K}]$", overlaying="y", side="right"),
)
fig.show()

## 2. The flame transfer function

The reference models the flame as a delayed second-order low-pass responder [3], the canonical shape for a V-shaped or swirl-stabilized flame: unit gain at zero frequency, a mild bump below the cutoff, and a phase that keeps winding as the pure lag and the filter's own phase add up.
Nefes ships it as `n_tau_lowpass2`, alongside the pure $n$–$\tau$ model and its first-order roll-off.

The value at the mode we are hunting is what decides the outcome.
Around $150\,$Hz the lag has already turned the response through about $110^\circ$ and the filter adds a further $60^\circ$, putting the heat release close to **antiphase** with the driving velocity.
A flame in antiphase absorbs acoustic energy rather than supplying it, which is why this configuration is the *stable* one.

In [ ]:
ftf = n_tau_lowpass2(FTF_GAIN, FTF_TAU, FTF_FC, FTF_ZETA)
freqs = np.linspace(0.0, 400.0, 400)

value = complex(ftf(OSCILOS_FREQ_HZ))
print(f"F at {OSCILOS_FREQ_HZ} Hz:  gain = {abs(value):.3f},  phase = {np.degrees(np.angle(value)):+.1f} deg")

plot_transfer_function(
    ftf, freqs, names=[rf"$f_c={FTF_FC:.0f}$ Hz, $\zeta={FTF_ZETA}$, $\tau={FTF_TAU*1e3:.0f}$ ms"],
    title=r"Flame transfer function (second-order low-pass with a time lag)",
).show()

## 3. The mode

A free mode is a complex root of $\det\mathbf{A}(\omega) = 0$, where
$\mathbf{A}(\omega) = \overline{\mathbf{J}} + \mathrm{i}\omega\mathbf{M} + \mathbf{P}(\omega) + \mathbf{S}(\omega)$
is the perturbation operator built on the converged mean state: the base Jacobian, the lumped storage, the duct propagation, and the flame's source.
`eigenmodes` finds the roots inside a chosen region of the complex plane by Beyn's contour-integral method, and certifies the count by the argument principle before it solves for them.

The modal frequency is $\mathrm{Re}(\omega)/2\pi$ and the growth rate is $-\mathrm{Im}(\omega)$; a negative growth rate is a decaying, stable mode.
We search the same band the reference's contour map covers.

In [ ]:
FREQ_BAND, GROWTH_BAND = (60.0, 320.0), (-160.0, 160.0)


def dominant_mode(solution, *, isentropic=True):
    """The most strongly growing mode in the search box, as ``(frequency [Hz], growth rate [1/s])``."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        res = eigenmodes(solution.problem, solution.x, freq_band=FREQ_BAND,
                         growth_band=GROWTH_BAND, isentropic=isentropic)
    if len(res) == 0:
        return np.nan, np.nan
    i = int(np.argmax(res.growth_rates))
    return float(res.freqs[i]), float(res.growth_rates[i])


with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    res = eigenmodes(sol.problem, sol.x, freq_band=FREQ_BAND, growth_band=GROWTH_BAND, isentropic=True)

f_nefes, g_nefes = float(res.freqs[0]), float(res.growth_rates[0])
print(f"{'':12s}{'frequency [Hz]':>16s}{'growth rate [1/s]':>20s}")
print(f"{'OSCILOS':12s}{OSCILOS_FREQ_HZ:>16.1f}{OSCILOS_GROWTH:>20.1f}")
print(f"{'Nefes':12s}{f_nefes:>16.1f}{g_nefes:>20.1f}")
print(f"{'difference':12s}{100*(f_nefes-OSCILOS_FREQ_HZ)/OSCILOS_FREQ_HZ:>15.1f}%"
      f"{100*(g_nefes-OSCILOS_GROWTH)/abs(OSCILOS_GROWTH):>19.1f}%")

res.plot_spectrum(title="EM2C combustor: certified spectrum in the search band").show()

The two codes agree to $0.5\,\%$ in frequency and $0.7\,\%$ in growth rate, and both place the mode on the stable side.
Section 7 shows that the residual is no larger than the reference's own ambiguity about its inlet temperature.

The energy ledger gives the same growth rate a second way, without the eigensolver.
For a free mode the acoustic-energy budget reads $2\sigma E = G + F_{\mathrm{in}}$: the growth rate times the stored energy equals the net power the interior elements produce plus the net flux the boundaries pass inward.
Solving it for $\sigma$ must return the contour eigenvalue.

In [ ]:
eb = res.energy_balance(0)
print(f"stored energy      E    = {eb.stored_energy:.3e}   (mode-scale units)")
print(f"interior net power G    = {eb.generation:+.3e}   (negative: the interior absorbs)")
print(f"boundary flux      F_in = {eb.boundary_flux:+.3e}")
print()
print(f"growth rate from the energy budget = {eb.growth_rate_energy:+.3f} 1/s")
print(f"growth rate from the eigenvalue    = {eb.growth_rate:+.3f} 1/s")
print(f"agree: {eb.consistent}")

## 4. Where the damping comes from

The mode decays at $-19\;\mathrm{s^{-1}}$ even though both terminals reflect with $|R| = 1$ and neither duct is lossy.
Two interior elements can move energy, and the network can be rebuilt with either of them switched off, which separates their contributions.

* The **dump plane**. The abrupt area increase carries the Borda–Carnot loss, and its linearization is dissipative to the acoustics. Replacing it with a lossless `isentropic_area_change` removes that sink.
* The **flame**. With `active=False` the heat release is steady and the flame is acoustically passive, retaining only its mean density jump.

In [ ]:
print(f"{'':>28s}{'frequency [Hz]':>16s}{'growth rate [1/s]':>20s}")
for expansion in ("borda", "isentropic"):
    for active in (False, True):
        f, g = dominant_mode(em2c(active=active, expansion=expansion))
        label = f"{expansion} dump, {'active' if active else 'passive'} flame"
        print(f"{label:>28s}{f:>16.1f}{g:>20.2f}")

The dump plane alone accounts for $-13\;\mathrm{s^{-1}}$ of the decay, and the antiphase flame adds roughly $-6\;\mathrm{s^{-1}}$ more; because the eigenproblem is nonlinear in $\omega$ the two do not add exactly.
The flame also **pulls the frequency up**, from $143\,$Hz to $153\,$Hz: its response is complex, so it is reactive as well as resistive, and the mode is not simply the passive resonance with damping bolted on.

With the dump made lossless and the flame passive, the residual growth rate is $+0.5\;\mathrm{s^{-1}}$ rather than zero.
That leftover is the mean flow: the flame's mean temperature jump produces $O(M)$ acoustic power, and the imposed $|R| = 1$ terminals are not exactly energy-neutral once the flow through them is retained (the neutral magnitudes are $(1\mp M)/(1\pm M)$).
It is more than an order of magnitude below the physical damping, which is the size $M \approx 10^{-2}$ calls for.

## 5. The entropy wave is a spectator here

The flame sheds an entropy wave as well as sound, and Nefes convects it: the full operator carries three characteristic families $(f, g, h)$, while `isentropic=True` drops the entropy wave and keeps only the two acoustic ones.

At this combustor the two give the **same eigenvalue**.
The reason is the outlet: a pressure-release end imposes $p' = 0$ on the acoustic waves and lets the entropy wave leave without converting any of it back into sound, so the wave never returns to the flame and cannot participate in a mode.
That is a statement about *this* boundary, not about entropy waves in general — end the same chamber in a **choked nozzle** and the entropy wave is converted back to sound by the Marble–Candel coupling, closing a feedback loop that can drive an instability with no acoustic path at all (see `indirect_noise_instability.ipynb`).

In [ ]:
f_isen, g_isen = dominant_mode(sol, isentropic=True)
f_full, g_full = dominant_mode(sol, isentropic=False)
print(f"{'isentropic (f, g only)':>26s}: f = {f_isen:8.3f} Hz   growth = {g_isen:+8.3f} 1/s")
print(f"{'full (f, g, h)':>26s}: f = {f_full:8.3f} Hz   growth = {g_full:+8.3f} 1/s")
print(f"\ndifference: {abs(f_full - f_isen):.1e} Hz, {abs(g_full - g_isen):.1e} 1/s "
      "-- the eigensolver's polish tolerance, not a physical difference")

## 6. The shape of the mode

The growth rate and frequency say *when* and *how fast*; the mode shape says *where*.
The pressure field is reconstructed analytically inside each cylinder — the duct's own phase relation evaluated at every interior station, not a discretization — and animated over one cycle.
The two terminals set its ends: a pressure antinode at the rigid inlet, a node at the open end.

The companion velocity field explains the flame's leverage.
The compact flame sits at $x = 234.3\,$mm, and that is where $|u'|$ *peaks*: the narrow injection unit concentrates the volume flow, so a transfer function of modest gain still moves a large amount of heat release.

In [ ]:
# Envelopes of the two fields, each scaled by its own maximum so the shapes can be read together.
fig = go.Figure()
for var, label, dash in (("p", r"$|\widehat{p}|$", "solid"), ("u", r"$|\widehat{u}|$", "dot")):
    path = res.field_along_network(0, variable=var)[0]
    x, amp = np.asarray(path.x), np.abs(np.asarray(path.values))
    fig.add_scatter(x=x, y=amp / amp.max(), mode="lines", name=label, line=dict(dash=dash, width=2.5))
fig.add_vline(x=sum(LENGTHS[:2]), line_width=1, line_dash="dash", line_color="#d62728",
              annotation_text="flame", annotation_position="top")
fig.update_layout(title="Mode shape: the flame sits at the velocity antinode",
                  xaxis_title=r"$x\;[\mathrm{m}]$", yaxis_title="amplitude (scaled)")
fig.show()

In [ ]:
res.animate_mode(0, variable="p", title=f"EM2C mode: p' along the combustor ({f_nefes:.0f} Hz)").show()

## 7. How close is close, and where does this case sit?

### The reference's own ambiguity

The report states $\overline{T}_1 = 300\,$K.
Its cold-tube case, however, states $300\,$K as well and then quotes $\overline{c} = 343.25\;\mathrm{m/s}$, which is the sound speed of air at $293.15\,$K, not at $300\,$K.
The inlet temperature that was actually used is therefore uncertain between the two, and the whole mean state scales with it.

Running the case at both values brackets the published pair: OSCILOS's $152.6\,$Hz and $-19.1\;\mathrm{s^{-1}}$ fall *between* the two Nefes answers.
The disagreement is the reference's ambiguity, not a discrepancy between the two codes' physics.

In [ ]:
print(f"{'inlet temperature':>22s}{'frequency [Hz]':>18s}{'growth rate [1/s]':>20s}")
for T_cold in (293.15, 300.0):
    f, g = dominant_mode(em2c(T_cold=T_cold))
    print(f"{T_cold:>19.2f} K{f:>18.2f}{g:>20.2f}")
print(f"{'OSCILOS':>22s}{OSCILOS_FREQ_HZ:>18.2f}{OSCILOS_GROWTH:>20.2f}")

### The stability band

The reference chose the chamber length so that the rig would be stable, and the flame lag $\tau = 2\,$ms lands just short of the instability band.
Sweeping the lag traces the band out: the growth rate crosses zero near $\tau \approx 2.6\,$ms, stays positive to $\tau \approx 5.4\,$ms, and drops back.

The published operating point therefore sits about half a millisecond of flame lag below the stability boundary.
That margin is small, and it is consistent with the rest of the reference: the same burner, given its *measured* flame describing function and a longer plenum and chamber, is reported unstable at $131\,$Hz in Sec. 4.2 of [2], where the experiment of [1] finds a limit cycle at $126\,$Hz.

In [ ]:
taus = np.linspace(0.0, 6.0e-3, 25)
growths = np.array([dominant_mode(em2c(tau=t))[1] for t in taus])

fig = go.Figure()
fig.add_scatter(x=taus * 1e3, y=growths, mode="lines+markers", name="Nefes")
fig.add_scatter(x=[FTF_TAU * 1e3], y=[OSCILOS_GROWTH], mode="markers", name="OSCILOS",
                marker=dict(size=13, symbol="star", color="#d62728"))
fig.add_hline(y=0.0, line_dash="dash", annotation_text="stability boundary")
fig.update_layout(
    title="Growth rate of the dominant mode versus the flame time lag",
    xaxis_title=r"$\text{flame time lag } \tau\;[\mathrm{ms}]$",
    yaxis_title=r"$\text{growth rate } \sigma\;[\mathrm{s^{-1}}]$",
)
fig.show()

## Summary

Nefes places the dominant mode of the EM2C combustor at $153.4\,$Hz with a growth rate of $-19.0\;\mathrm{s^{-1}}$, against the $152.6\,$Hz and $-19.1\;\mathrm{s^{-1}}$ that OSCILOS reports for the same configuration, and the residual is bounded by the reference's own uncertainty about its inlet temperature.

What the check covers is the whole chain, not one layer of it: the mean flow is solved on the network from a cold start rather than prescribed section by section, the perturbation operator is the linearization of the same residuals evaluated at that converged state, and the mode is extracted from the operator with a certified count.
Along the way the run also answers questions the eigenvalue alone does not — that the dump plane, not the flame, is the main sink of acoustic energy; that the flame is reactive as well as resistive, pulling the frequency up by $10\,$Hz; that the entropy wave the flame sheds is a spectator at a pressure-release outlet; and that the published operating point sits half a millisecond of flame lag away from an instability.

This case is pinned as a regression in `tests/test_oscilos_em2c.py`, and is written up in the [benchmarks](../../docs/validation/benchmarks.md) document.

## References

1. P. Palies, D. Durox, T. Schuller and S. Candel, "Nonlinear combustion instability analysis based on the flame describing function applied to turbulent premixed swirling flames," *Combustion and Flame* **158**(10), 1980–1991 (2011). [doi:10.1016/j.combustflame.2011.02.012](https://doi.org/10.1016/j.combustflame.2011.02.012)
2. J. Li, D. Yang, C. Luzzato and A. S. Morgans, *Open Source Combustion Instability Low Order Simulator (OSCILOS–Long) Technical Report*, Department of Mechanical Engineering, Imperial College London (2017). The case reproduced here is Sec. 5.4.1; the reflection coefficients and mean-flow conventions are Secs. 2 and 3. Report and code: [oscilos.com](https://www.oscilos.com), [github.com/MorgansLab/OSCILOS_long](https://github.com/MorgansLab/OSCILOS_long).
3. A. P. Dowling, "Nonlinear self-excited oscillations of a ducted flame," *Journal of Fluid Mechanics* **346**, 271–290 (1997). [doi:10.1017/S0022112097006484](https://doi.org/10.1017/S0022112097006484) — the second-order low-pass flame transfer function.